# 1) Title and objective

## OH Leuven Attendance Prediction

This notebook is the single project entry point for academic submission. It runs the existing production pipeline from `src/` and reports only the final `xgboost_log` model.

No manual feature engineering or manual training logic is implemented in this notebook.


## 2) Imports


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_DATA_DIR
from src.train import run_pipeline
from src.predict import predict_new_matches

pd.set_option("display.max_columns", None)


## 3) Load data path


In [ ]:
DATA_DIR = Path(DEFAULT_DATA_DIR)
print(f"Data directory: {DATA_DIR}")


## 4) Run full pipeline (reuse existing code)


In [ ]:
result = run_pipeline(
    data_dir=DATA_DIR,
)
print("Pipeline run completed.")


## 5) Extract best model and predictions


In [ ]:
TARGET_MODEL = "xgboost_log"

run_info = result["run_info"]
predictions_df = result["predictions"].copy()

if TARGET_MODEL not in run_info.get("metrics_by_model", {}):
    raise ValueError(f"{TARGET_MODEL} not found in pipeline output metrics.")

pred_col = f"pred_{TARGET_MODEL}"
if pred_col not in predictions_df.columns:
    raise ValueError(f"Prediction column missing: {pred_col}")

y_test = predictions_df["actual"].to_numpy(dtype=float)
y_pred = predictions_df[pred_col].to_numpy(dtype=float)

print(f"Pipeline-selected best model: {run_info.get('best_model_name')}")
print(f"Notebook reporting model: {TARGET_MODEL}")


## 6) Show metrics (MAE, RMSE, MAPE, R2)


In [ ]:
metrics = run_info["metrics_by_model"][TARGET_MODEL]
metrics_df = pd.DataFrame(
    [
        {
            "Model": TARGET_MODEL,
            "MAE": metrics["mae"],
            "RMSE": metrics["rmse"],
            "MAPE": metrics["mape"],
            "R2": metrics["r2"],
        }
    ]
)
display(metrics_df)


## 7) Plot actual vs predicted


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.8)
line_min = float(min(y_test.min(), y_pred.min()))
line_max = float(max(y_test.max(), y_pred.max()))
plt.plot([line_min, line_max], [line_min, line_max], color="red", linewidth=1.5)
plt.xlabel("Actual attendance")
plt.ylabel("Predicted attendance")
plt.title(f"Actual vs Predicted ({TARGET_MODEL})")
plt.tight_layout()
plt.show()


## 8) Feature importance (top 10)


In [ ]:
feature_importance_df = result["feature_importance"].copy().head(10)
display(feature_importance_df)

plot_df = feature_importance_df.sort_values("importance", ascending=True)
plt.figure(figsize=(8, 5))
plt.barh(plot_df["feature"], plot_df["importance"])
plt.xlabel("Importance")
plt.title(f"Top 10 feature importance ({TARGET_MODEL})")
plt.tight_layout()
plt.show()


## 9) Future prediction example (existing prediction pipeline)


In [ ]:
new_matches = pd.DataFrame(
    [
        {
            "match_date": "2026-09-12",
            "away_team": "Club Brugge",
            "stage": "Regular Season",
            "kickoff_time": "20:45:00",
        }
    ]
)

future_predictions_df, future_summary = predict_new_matches(
    new_matches_df=new_matches,
    data_dir=DATA_DIR,
)

display(future_predictions_df)
display(pd.DataFrame([future_summary]))


## 10) Short conclusion

This final notebook keeps the workflow short, readable, and reproducible by calling the existing `src/` pipeline directly. The reported model is `xgboost_log`, with metrics and predictions produced by the same validated training and inference code used in the project.
